# M2: EV Charging Points — National Baseline
## IE Sustainability Datathon March 2026, Iberdrola Challenge

**Objective:** Parse the national EV charging points dataset (DATEX2 XML), filter to interurban stations only, and produce `total_existing_stations_baseline` for File 1.csv.

**Data source:** DGT / MITERD via National Access Point (NAP)  
→ Format: DATEX2 v3 XML (~80 MB, updated every 24 hours)  
→ File: `Data/raw/ev_charging_points_national/electrolineras_spain.xml`

**Depends on:**
- `Data/raw/ev_charging_points_national/electrolineras_spain.xml` — run `Data/scripts/download_ev_charging_national.sh` if missing
- `Data/raw/road_routes_spain/carreteras_RTIG.geojson` — output of **M1** (required for Section 5 interurban filter)

**Outputs:**
- `Data/interim/m2_charging_sites_all.csv` — all parsed sites (pre-filter)
- `Data/interim/m2_charging_sites_interurban.csv` — interurban sites only
- `total_existing_stations_baseline` → used in **File 1.csv**

---
**Fork reference (mandatory):** https://github.com/NOSIEMPRE/Laboratorio-de-Datos

## 0. Setup: Dependencies & Paths

In [ ]:
import os
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point

# ── Paths ─────────────────────────────────────────────────────────────────────
XML_PATH       = '../Data/raw/ev_charging_points_national/electrolineras_spain.xml'
RTIG_PATH      = '../Data/raw/road_routes_spain/carreteras_RTIG.geojson'
OUT_ALL        = '../Data/interim/m2_charging_sites_all.csv'
OUT_INTERURBAN = '../Data/interim/m2_charging_sites_interurban.csv'
os.makedirs('../Data/interim', exist_ok=True)

# ── DATEX2 v3 namespaces ──────────────────────────────────────────────────────
NS = {
    'd2' : 'http://datex2.eu/schema/3/d2Payload',
    'com': 'http://datex2.eu/schema/3/common',
    'loc': 'http://datex2.eu/schema/3/locationReferencing',
    'egi': 'http://datex2.eu/schema/3/energyInfrastructure',
    'fac': 'http://datex2.eu/schema/3/facilities',
    'locx': 'http://datex2.eu/schema/3/locationExtension',
}

# ── Verify input file ─────────────────────────────────────────────────────────
assert os.path.exists(XML_PATH), (
    f'XML not found: {XML_PATH}\n'
    'Run: bash Data/scripts/download_ev_charging_national.sh'
)
size_mb = os.path.getsize(XML_PATH) / 1e6
print(f'XML file   : {XML_PATH}')
print(f'Size       : {size_mb:.1f} MB')
print('Dependencies loaded.')

## 1. Parse XML

The DATEX2 v3 structure is:
```
energyInfrastructureSite  ← one physical location (= one "station" for our count)
  └── locationReference/coordinatesForDisplay/latitude + longitude
  └── fac:name/values/value              ← site name
  └── fac:operator/name/values/value     ← operator name
  └── energyInfrastructureStation
        └── refillPoint  [1..N]          ← one charging point per connector group
              └── connector
                    └── connectorType
                    └── maxPowerAtSocket  (watts → divide by 1000 for kW)
```

For `total_existing_stations_baseline` we count **sites** (not connectors), consistent with how "charging stations" are counted in infrastructure planning.

In [ ]:
def _text(el, tag):
    """Return stripped text of first matching sub-element, or None."""
    found = el.find(tag, NS)
    return found.text.strip() if found is not None and found.text else None


def parse_site(site):
    """Extract one row dict from an energyInfrastructureSite element."""
    row = {}
    row['site_id'] = site.get('id', '')

    # ── Name ──────────────────────────────────────────────────────────────────
    row['name'] = _text(site, 'fac:name/com:values/com:value')

    # ── Coordinates (WGS84) ───────────────────────────────────────────────────
    lat_el = site.find('.//loc:coordinatesForDisplay/loc:latitude',  NS)
    lon_el = site.find('.//loc:coordinatesForDisplay/loc:longitude', NS)
    try:
        row['latitude']  = float(lat_el.text) if lat_el is not None else None
        row['longitude'] = float(lon_el.text) if lon_el is not None else None
    except (ValueError, TypeError):
        row['latitude'] = row['longitude'] = None

    # ── Operator ──────────────────────────────────────────────────────────────
    row['operator'] = _text(site, 'fac:operator/fac:name/com:values/com:value')

    # ── Refill points (charging connectors) ───────────────────────────────────
    refill_points = site.findall('.//egi:refillPoint', NS)
    row['n_refill_points'] = len(refill_points)

    # Max power across all connectors (watts → kW)
    powers = []
    connector_types = set()
    for rp in refill_points:
        for conn in rp.findall('.//egi:connector', NS):
            pwr = _text(conn, 'egi:maxPowerAtSocket')
            if pwr:
                try:
                    powers.append(float(pwr) / 1000)
                except ValueError:
                    pass
            ct = _text(conn, 'egi:connectorType')
            if ct:
                connector_types.add(ct)

    row['max_power_kw']     = max(powers) if powers else None
    row['connector_types']  = '|'.join(sorted(connector_types)) if connector_types else None

    # ── Site type (openSpace, building, etc.) ─────────────────────────────────
    row['type_of_site'] = _text(site, 'egi:typeOfSite')

    return row


print('Parsing XML ... (may take ~30 seconds for 12k sites)')
tree = ET.parse(XML_PATH)
root = tree.getroot()

records = []
for site in root.iter(f"{{{NS['egi']}}}energyInfrastructureSite"):
    records.append(parse_site(site))

df = pd.DataFrame(records)
print(f'Sites parsed   : {len(df):,}')
print(f'Columns        : {df.columns.tolist()}')
print(f'Null latitude  : {df["latitude"].isna().sum()}')
print(f'Null longitude : {df["longitude"].isna().sum()}')
print()
print(df.head(3).to_string())

## 2. Clean & Build GeoDataFrame

In [ ]:
# Drop sites with no coordinates (can't be spatially matched)
n_before = len(df)
df = df.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
n_dropped = n_before - len(df)
if n_dropped:
    print(f'Dropped {n_dropped} sites with missing coordinates.')

# Sanity-check: coordinates inside Spain's bounding box
# (includes Canary Islands and Balearics)
mask_spain = (
    df['longitude'].between(-20, 5) &
    df['latitude'].between(27, 45)
)
n_out = (~mask_spain).sum()
if n_out:
    print(f'WARNING: {n_out} sites outside Spain bounding box — dropping.')
    df = df[mask_spain].reset_index(drop=True)

# Build GeoDataFrame (WGS84)
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitude'], df['latitude']),
    crs='EPSG:4326'
)

print(f'Sites with valid coordinates : {len(gdf):,}')
print(f'CRS                          : {gdf.crs}')
print()
print('Power distribution (kW):')
print(gdf['max_power_kw'].describe().round(1).to_string())

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── 1. Power distribution ─────────────────────────────────────────────────────
ax = axes[0, 0]
bins = [0, 7.4, 22, 50, 100, 150, 250, 350, 1000]
labels = ['≤7.4 (AC slow)', '7.4–22 (AC)', '22–50 (AC fast)',
          '50–100 (DC)', '100–150 (DC fast)', '150–250 (DC HPC)',
          '250–350 (DC HPC+)', '>350 (Ultra)']
gdf['power_band'] = pd.cut(gdf['max_power_kw'], bins=bins, labels=labels)
power_counts = gdf['power_band'].value_counts().sort_index()
power_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Sites by Max Power Band (kW)')
ax.set_xlabel('Number of sites')

# ── 2. Top 15 operators ───────────────────────────────────────────────────────
ax = axes[0, 1]
top_ops = gdf['operator'].value_counts().head(15)
top_ops.plot(kind='barh', ax=ax, color='darkorange')
ax.set_title('Top 15 Operators by Site Count')
ax.set_xlabel('Number of sites')
ax.invert_yaxis()

# ── 3. Refill points per site ─────────────────────────────────────────────────
ax = axes[1, 0]
gdf['n_refill_points'].clip(upper=20).value_counts().sort_index().plot(
    kind='bar', ax=ax, color='forestgreen'
)
ax.set_title('Number of Charging Points per Site')
ax.set_xlabel('Charging points (capped at 20)')
ax.set_ylabel('Sites')

# ── 4. Geographic distribution ────────────────────────────────────────────────
ax = axes[1, 1]
gdf.plot(ax=ax, markersize=1, color='crimson', alpha=0.4)
ax.set_title(f'All EV Charging Sites in Spain (n={len(gdf):,})')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

print(f'\nTotal sites: {len(gdf):,}')
print(f'Sites with DC fast charging (≥50 kW): {(gdf["max_power_kw"] >= 50).sum():,}')
print(f'Sites with HPC (≥150 kW): {(gdf["max_power_kw"] >= 150).sum():,}')

## 4. Save All Sites (Pre-filter)

Save the full parsed dataset before applying the interurban filter. This is used as a reference and for downstream validation.

In [ ]:
save_cols = ['site_id', 'name', 'latitude', 'longitude',
             'operator', 'n_refill_points', 'max_power_kw',
             'connector_types', 'type_of_site']

gdf[save_cols].to_csv(OUT_ALL, index=False)
print(f'Saved all sites → {OUT_ALL}')
print(f'Rows: {len(gdf):,}  |  Columns: {save_cols}')

## 5. Interurban Filter (requires M1 output)

Filter to stations that lie within **500 m** of the eligible RTIG road network (`carreteras_RTIG.geojson`).

**Methodology (documented for Analytical Report — criterion T1):**
1. Load the RTIG road network (WGS84)
2. Reproject both layers to ETRS89 UTM Zone 30N (EPSG:25830) for metric distance operations
3. Buffer road lines by **500 m** — captures stations at motorway service areas, which are typically set back from the road centreline
4. Spatial join: keep only charging sites whose point geometry intersects the buffered road layer
5. Drop duplicates from the join (a site may intersect multiple road segments)

**500 m buffer justification:** Motorway service areas in Spain are typically 50–300 m from the road centreline. A 500 m buffer is conservative enough to capture all legitimate roadside sites while excluding urban stations in nearby towns.

> ⚠️ **Requires M1 output.** If `carreteras_RTIG.geojson` does not exist, run `M1_Road_Network_RTIG.ipynb` first.

In [ ]:
BUFFER_M = 500  # metres

assert os.path.exists(RTIG_PATH), (
    f'M1 output not found: {RTIG_PATH}\n'
    'Run M1_Road_Network_RTIG.ipynb first.'
)

# ── Load RTIG road network ────────────────────────────────────────────────────
roads = gpd.read_file(RTIG_PATH)   # already EPSG:4326
print(f'RTIG road segments loaded: {len(roads):,}')

# ── Reproject both layers to metric CRS for buffering ────────────────────────
METRIC_CRS = 'EPSG:25830'
roads_m  = roads.to_crs(METRIC_CRS)
sites_m  = gdf.to_crs(METRIC_CRS)

# ── Buffer road lines ─────────────────────────────────────────────────────────
print(f'Buffering road network by {BUFFER_M} m ...')
roads_buffer = roads_m.copy()
roads_buffer['geometry'] = roads_m.geometry.buffer(BUFFER_M)
road_union = roads_buffer.geometry.union_all()   # single polygon for fast sjoin
print('Buffer created.')

# ── Spatial join: keep sites inside the buffer ────────────────────────────────
gdf_interurban = sites_m[sites_m.geometry.within(road_union)].copy()
gdf_interurban = gdf_interurban.to_crs('EPSG:4326')  # back to WGS84

n_total      = len(gdf)
n_interurban = len(gdf_interurban)
pct          = n_interurban / n_total * 100

print()
print('=' * 50)
print('  INTERURBAN FILTER RESULTS')
print('=' * 50)
print(f'  Total sites (national)   : {n_total:>6,}')
print(f'  Interurban sites (≤500m) : {n_interurban:>6,}  ({pct:.1f}%)')
print(f'  Excluded (urban/other)   : {n_total - n_interurban:>6,}')
print('=' * 50)

## 6. Visualise Interurban Sites

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ── Left: all sites vs interurban ─────────────────────────────────────────────
ax = axes[0]
gdf.plot(ax=ax, markersize=1, color='lightgrey', alpha=0.5, label='All sites')
gdf_interurban.plot(ax=ax, markersize=3, color='steelblue', alpha=0.8, label='Interurban')
roads.plot(ax=ax, linewidth=0.3, color='darkorange', alpha=0.4, label='RTIG roads')
ax.set_title(f'All ({len(gdf):,}) vs Interurban ({n_interurban:,}) Charging Sites')
ax.legend(fontsize=8)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# ── Right: interurban sites coloured by max power ─────────────────────────────
ax = axes[1]
roads.plot(ax=ax, linewidth=0.3, color='lightgrey', alpha=0.5)

power_bins   = [0, 50, 150, float('inf')]
power_labels = ['<50 kW (AC/DC slow)', '50–150 kW (DC fast)', '≥150 kW (HPC)']
power_colors = ['gold', 'darkorange', 'crimson']

gdf_interurban['power_cat'] = pd.cut(
    gdf_interurban['max_power_kw'],
    bins=power_bins, labels=power_labels
)

handles = []
for label, color in zip(power_labels, power_colors):
    subset = gdf_interurban[gdf_interurban['power_cat'] == label]
    subset.plot(ax=ax, markersize=4, color=color, alpha=0.85)
    handles.append(mpatches.Patch(color=color, label=f'{label} ({len(subset)})'))

ax.set_title('Interurban Sites by Max Power')
ax.legend(handles=handles, fontsize=8, loc='lower left')
ax.set_xlabel('Longitude')

plt.tight_layout()
plt.show()

## 7. Compute `total_existing_stations_baseline` & Export

In [ ]:
# The baseline count = number of unique interurban sites
total_existing_stations_baseline = int(gdf_interurban['site_id'].nunique())

print('=' * 50)
print('  BASELINE KPI (for File 1.csv)')
print('=' * 50)
print(f'  total_existing_stations_baseline = {total_existing_stations_baseline}')
print('=' * 50)
print()
print('Interurban site breakdown:')
print(f'  Sites with DC fast (≥50 kW)  : {(gdf_interurban["max_power_kw"] >= 50).sum()}')
print(f'  Sites with HPC (≥150 kW)     : {(gdf_interurban["max_power_kw"] >= 150).sum()}')
print(f'  Avg charging points per site : {gdf_interurban["n_refill_points"].mean():.1f}')
print()

# Save interurban sites CSV
gdf_interurban[save_cols].to_csv(OUT_INTERURBAN, index=False)
print(f'Saved interurban sites → {OUT_INTERURBAN}')

# Save KPI as JSON for File 1 assembly
import json
os.makedirs('../Data/processed', exist_ok=True)
kpi_path = '../Data/processed/m2_baseline.json'
with open(kpi_path, 'w') as f:
    json.dump({'total_existing_stations_baseline': total_existing_stations_baseline}, f, indent=2)
print(f'Saved KPI → {kpi_path}')

## 8. Output Verification

Per evaluation criterion T5, all output file structures must be printed and verified.

In [ ]:
df_v = pd.read_csv(OUT_INTERURBAN)

print('OUTPUT VERIFICATION — m2_charging_sites_interurban.csv')
print('-' * 60)
print(f'  Rows    : {len(df_v):,}')
print(f'  Columns : {df_v.columns.tolist()}')
print()
print(df_v.dtypes.to_string())
print()
print('Sample (first 3 rows):')
print(df_v.head(3).to_string(index=False))
print()
print(f'  Lat range  : {df_v["latitude"].min():.3f} → {df_v["latitude"].max():.3f}')
print(f'  Lon range  : {df_v["longitude"].min():.3f} → {df_v["longitude"].max():.3f}')
print()

# ── Assertions ────────────────────────────────────────────────────────────────
assert len(df_v) > 0, 'ERROR: interurban file is empty'
assert df_v['latitude'].between(27, 45).all(), 'ERROR: latitude out of Spain range'
assert df_v['longitude'].between(-20, 5).all(), 'ERROR: longitude out of Spain range'
assert df_v['site_id'].nunique() == len(df_v), 'ERROR: duplicate site IDs'
assert total_existing_stations_baseline > 0, 'ERROR: baseline count must be positive'

print('✓ Row count check passed')
print('✓ Coordinate bounds check passed (Spain)')
print('✓ No duplicate site IDs')
print()
print(f'→ total_existing_stations_baseline = {total_existing_stations_baseline}')
print(f'→ Ready for File 1.csv assembly')